# Assignment 3.2 - Support Vector Machines (SVM)

Please submit your solution of this notebook in the Whiteboard at the corresponding Assignment entry as .ipynb-file and as .pdf. <br><br>
Please do **NOT** rename the file!

#### State both names of your group members here:
[Jane and John Doe]

In [6]:
[Hamna Rashid Ahmed and Khurram Ameer Randhawa]

---

## Grading Info/Details - Assignment 3.2:

The assignment will be graded semi-automatically, which means that your code will be tested against a set of predefined test cases and qualitatively assessed by a human. This will speed up the grading process for us.

* For passing the test scripts:
    - Please make sure to **NOT** alter predefined class or function names, as this would lead to failing of the test scripts.
    - Please do **NOT** rename the files before uploading to the Whiteboard!

* **(RESULT)** tags indicate checkpoints that will be specifically assessed by a human.

* You will pass the assignment if you pass the majority of test cases and we can at least confirm effort regarding the **(RESULT)**-tagged checkpoints per task.

---

## Task 3.2.1 - SVM

Implement a Support Vector Machine (SVM) classifier from scratch using Stochastic Gradient Descent (SGD) algorithm for training.

* Implement the `SVM` class below. It should hold logic for an SVM classifier using the Hinge loss. **(RESULT)**
* Feel free to test is using synthetic data first (i.e. `make_blobs` and `make_moons` for linearly separable and non-separable tasks).
* Finally, run your final SVM implementation of the [Wine Dataset](https://archive.ics.uci.edu/dataset/109/wine) dataset. You may use sklearns functions to load the dataset. **(RESULT)** <br> `from sklearn.datasets import load_wine`

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine, make_blobs, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [8]:


# --- Utility Functions ---
def calculate_accuracy(y_true, y_pred):
    """Calculates the classification accuracy."""
    return np.mean(y_true == y_pred)

# --- Task 3.2.1 - Linear SVM Implementation ---

class SVM:
    """
    Support Vector Machine classifier implemented from scratch using NumPy.
    Uses Stochastic Gradient Descent (SGD) to optimize the hinge loss with L2 regularization.
    """

    def __init__(self, learning_rate=0.0001, lambda_param=0.01, n_iters=1000):
        """
        Initializes the SVM with hyperparameters.
        :param learning_rate: The step size for gradient descent.
        :param lambda_param: The regularization strength (lambda).
        :param n_iters: The number of training iterations (epochs).
        """
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.w = None  # Weights vector
        self.b = None  # Bias scalar

    def fit(self, X, y):
        """
        Trains the SVM model using Stochastic Gradient Descent.
        (RESULT)
        :param X: Training features (n_samples, n_features).
        :param y: Target labels (n_samples,).
        """
        n_samples, n_features = X.shape

        # Convert labels from {0, 1, 2, ...} to {-1, 1} for binary classification
        # Since Wine has 3 classes, we simplify for the linear SVM to the first two classes
        # For a true multi-class SVM, one-vs-all or one-vs-one approach is needed.
        # Here, we treat the first class (0) as 1 and the others (1, 2) as -1
        y_mod = np.where(y == 0, 1, -1)

        # Initialize weights and bias to zeros
        self.w = np.zeros(n_features)
        self.b = 0

        for _ in range(self.n_iters):
            # Select a random sample for Stochastic Gradient Descent (SGD)
            i = np.random.randint(n_samples)
            x_i = X[i]
            y_i = y_mod[i]

            # Calculate the condition: y_i * (w.x_i + b)
            condition = y_i * (np.dot(x_i, self.w) + self.b)

            if condition >= 1:
                # Correctly classified or on the margin: update only for regularization
                dw = self.lambda_param * self.w
                db = 0
            else:
                # Misclassified or inside the margin: update for regularization and hinge loss
                dw = self.lambda_param * self.w - y_i * x_i
                db = -y_i

            # Update weights and bias
            self.w -= self.lr * dw
            self.b -= self.lr * db

    def predict(self, X):
        """
        Predicts the class labels for the given features.
        :param X: Features (n_samples, n_features).
        :return: Predicted labels (1 or -1).
        """
        approx = np.dot(X, self.w) + self.b
        # The sign of the result gives the class: sign(approx) -> {1, -1}
        return np.sign(approx)


## Task 3.2.2 - SVM with Kernel Trick

* Extend your SVM implementation to support the kernel trick with the Radial Basis Function (RBF) kernel. Report on the performance on the [Wine Dataset](https://archive.ics.uci.edu/dataset/109/wine) dataset and compare to the linear SVM. **(RESULT)**

In [9]:
from sklearn.datasets import make_blobs, make_moons
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_wine

In [10]:
class KernelSVM:
    """
    Support Vector Machine with Kernel Trick implementation (Dual form - simplified iterative solution).
    Uses the Radial Basis Function (RBF) kernel.
    """

    def __init__(self, C=1.0, gamma=0.1, n_iters=100):
        """
        Initializes the KernelSVM with hyperparameters.
        :param C: Regularization parameter (misclassification penalty).
        :param gamma: Parameter for the RBF kernel.
        :param n_iters: The number of training iterations.
        """
        self.C = C
        self.gamma = gamma
        self.n_iters = n_iters
        self.alphas = None  # Lagrange multipliers (Dual coefficients)
        self.X_train = None # Store training data for kernel calculation
        self.y_train_mod = None # Stored modified labels
        self.b = 0

    def _kernel_function(self, X1, X2):
        """
        Compute the Radial Basis Function (RBF) kernel between two matrices.
        K(xi, xj) = exp(-gamma * ||xi - xj||^2)
        :param X1: Matrix 1.
        :param X2: Matrix 2.
        :return: Kernel matrix.
        """
        if X1.ndim == 1 and X2.ndim == 1:
            # Case for two single vectors
            return np.exp(-self.gamma * np.linalg.norm(X1 - X2)**2)

        # Calculate pairwise squared Euclidean distances (X1 is (N, D), X2 is (M, D))
        # Equivalent to: ||X1[i] - X2[j]||^2
        dist_sq = np.sum(X1**2, axis=1, keepdims=True) + np.sum(X2**2, axis=1) - 2 * np.dot(X1, X2.T)

        # Apply RBF kernel formula
        return np.exp(-self.gamma * dist_sq)

    def fit(self, X, y):
        """
        Trains the Kernel SVM model using a simplified iterative dual update (similar to Kernel Perceptron).
        (RESULT)
        :param X: Training features (n_samples, n_features).
        :param y: Target labels (n_samples,).
        """
        n_samples = X.shape[0]
        self.X_train = X

        # Convert labels from {0, 1, 2, ...} to {-1, 1}
        # Again, we simplify to binary classification (Class 0 = 1, others = -1)
        self.y_train_mod = np.where(y == 0, 1, -1)

        # Initialize alpha (Lagrange Multipliers) to zeros
        self.alphas = np.zeros(n_samples)

        # Pre-compute the Gram matrix (Kernel Matrix on training data)
        K = self._kernel_function(X, X)

        for _ in range(self.n_iters):
            # Iterate through all training samples (Stochastic or full batch update is possible)
            for i in range(n_samples):
                # Calculate the decision function output for the current sample x_i
                # Output f(x_i) = sum(alpha_j * y_j * K(x_j, x_i)) + b
                prediction_output = np.sum(self.alphas * self.y_train_mod * K[:, i]) + self.b

                # Check for misclassification (similar to hinge loss check, but simplified)
                if self.y_train_mod[i] * prediction_output <= 0: # Misclassified
                    # Simplified update rule (Perceptron-style)
                    self.alphas[i] += 1
                    # Clip alpha to the range [0, C] (essential for SVM)
                    self.alphas[i] = np.clip(self.alphas[i], 0, self.C)

        # Calculate bias (b) using the support vectors (alpha > 0 and alpha < C, but we simplify)
        # Using a simple mean calculation over all samples for approximation:
        # b = mean(y - sum(alpha_j * y_j * K(x_j, x)))
        decision_values = np.sum(self.alphas * self.y_train_mod * K, axis=1)
        self.b = np.mean(self.y_train_mod - decision_values)

        # Identify and store support vectors (optional for performance, but good practice)
        sv_indices = np.where(self.alphas > 1e-5)[0]
        self.support_vectors = self.X_train[sv_indices]
        self.sv_alphas = self.alphas[sv_indices]
        self.sv_y = self.y_train_mod[sv_indices]


    def predict(self, X):
        """
        Predicts the class labels for the given features using the kernel trick.
        :param X: Features (n_samples, n_features).
        :return: Predicted labels (1 or -1).
        """
        # Calculate the decision function: f(x) = sum(alpha_i * y_i * K(x_i, x)) + b

        # Kernel matrix between training data (SV) and new data (X)
        K_pred = self._kernel_function(self.X_train, X)

        # Summation over all training samples (using self.alphas, self.y_train_mod)
        # np.dot((self.alphas * self.y_train_mod).T, K_pred)
        decision_function = np.dot((self.alphas * self.y_train_mod), K_pred) + self.b

        # Return the sign of the decision function
        return np.sign(decision_function)


In [11]:
# 1. Load Data, Preprocess, and Split
wine = load_wine()
X, y = wine.data, wine.target


# Standardize the features (Essential for SVM, especially RBF kernel)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# For simplification in scratch implementation, we perform binary classification (Class 0 vs. Class 1/2)
# The scratch SVM classes above are tailored for -1/+1 output.
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

# Convert original 3-class target for evaluation of the custom binary SVM:
y_test_binary = np.where(y_test == 0, 1, -1)


# 2. Test Linear SVM (Task 3.2.1)
print("---------------------------------------")
print("1. Linear SVM (SGD) on Wine Dataset")
print("---------------------------------------")
linear_svm = SVM(learning_rate=0.0001, lambda_param=0.01, n_iters=5000)

# Fit the model
linear_svm.fit(X_train, y_train)

# Predict on the test set
y_pred_linear = linear_svm.predict(X_test)

# Evaluate (RESULT)
accuracy_linear = calculate_accuracy(y_test_binary, y_pred_linear)
print(f"Test Accuracy (Linear SVM): {accuracy_linear * 100:.2f}%")
print("(RESULT: Linear SVM Performance)")


# 3. Test Kernel SVM (Task 3.2.2)
print("\n---------------------------------------")
print("2. Kernel SVM (RBF) on Wine Dataset")
print("---------------------------------------")

# Parameters tuned for RBF kernel on this data:
C_param = 1.0
gamma_param = 0.01
n_iters_kernel = 50

kernel_svm = KernelSVM(C=C_param, gamma=gamma_param, n_iters=n_iters_kernel)

# Fit the model
kernel_svm.fit(X_train, y_train)

# Predict on the test set
y_pred_kernel = kernel_svm.predict(X_test)

# Evaluate (RESULT)
accuracy_kernel = calculate_accuracy(y_test_binary, y_pred_kernel)
print(f"Test Accuracy (Kernel SVM): {accuracy_kernel * 100:.2f}%")

print("\n(RESULT: Kernel SVM Performance Comparison)")
print("Comparison:")
print(f"- Linear SVM Accuracy: {accuracy_linear * 100:.2f}%")
print(f"- Kernel SVM Accuracy: {accuracy_kernel * 100:.2f}%")
print("The Kernel SVM typically performs better on non-linearly separable data due to the RBF kernel mapping features to a higher-dimensional space.")

---------------------------------------
1. Linear SVM (SGD) on Wine Dataset
---------------------------------------
Test Accuracy (Linear SVM): 94.44%
(RESULT: Linear SVM Performance)

---------------------------------------
2. Kernel SVM (RBF) on Wine Dataset
---------------------------------------
Test Accuracy (Kernel SVM): 100.00%

(RESULT: Kernel SVM Performance Comparison)
Comparison:
- Linear SVM Accuracy: 94.44%
- Kernel SVM Accuracy: 100.00%
The Kernel SVM typically performs better on non-linearly separable data due to the RBF kernel mapping features to a higher-dimensional space.


## Congratz, you made it! :)